# **end-to-end tutorial on PostgreSQL**

## **1. Connecting & Navigating via Terminal (`psql`)**

PostgreSQL uses the interactive `psql` command-line tool. Unlike MySQL or SQLite, you cannot use the `USE` SQL command to change databases; instead, you use terminal meta-commands.

* **Log in to PostgreSQL:**
```bash
psql -U postgres
```


* **List all databases:**
```sql
\l
```


* **Switch to a database:**
```sql
\c dataengineering
```


* **List all tables in the current database/schema:**
```sql
\dt chicago_dmv.*
```


* **Exit `psql`:**
```sql
\q
```



---

## **2. Setting Up Databases and Schemas**

Before creating tables, you organize your database using schemas (namespaces that group related tables together).

```sql
-- Create a new database (run outside psql or via template)
-- CREATE DATABASE dataengineering;

-- Create a schema to keep your tables organized
CREATE SCHEMA IF NOT EXISTS chicago_dmv;
```

---

## **3. Creating Tables (DDL)**

Data Definition Language (DDL) controls the structure of your data. Here are the core tables you designed, formatted correctly with proper data types and constraints:

```sql
-- 1. Vehicle Table
CREATE TABLE chicago_dmv.Vehicle (
    CRASH_UNIT_ID integer,
    CRASH_ID text,
    CRASH_DATE timestamp,
    VEHICLE_ID integer,
    VEHICLE_MAKE text,
    VEHICLE_MODEL text,
    VEHICLE_YEAR integer,
    VEHICLE_TYPE text
);

-- 2. Person Table
CREATE TABLE chicago_dmv.Person (
    PERSON_ID text,
    CRASH_ID text,
    CRASH_DATE timestamp,
    PERSON_TYPE text,
    VEHICLE_ID integer,
    PERSON_SEX text,
    PERSON_AGE integer
);

-- 3. Crash Time Table (Renamed from 'Time' to avoid reserved keyword conflicts)
CREATE TABLE chicago_dmv.Crash_Time (
    CRASH_DATE timestamp,
    CRASH_ID text,
    CRASH_HOUR integer,
    CRASH_DAY_OF_WEEK integer,
    CRASH_MONTH integer,
    DATE_POLICE_NOTIFIED timestamp
);

-- 4. Crash Table
CREATE TABLE chicago_dmv.Crash (
    CRASH_UNIT_ID integer,
    CRASH_ID text,
    PERSON_ID text,
    VEHICLE_ID integer,
    NUM_UNITS numeric,
    TOTAL_INJURIES numeric
);

```

---

## **4. Inserting & Updating Data (DML)**

Data Manipulation Language (DML) manages the records inside your tables.

* **Inserting Data:**
```sql
INSERT INTO chicago_dmv.Vehicle (CRASH_UNIT_ID, CRASH_ID, VEHICLE_MAKE, VEHICLE_YEAR)
VALUES (101, 'C123456', 'Toyota', 2018);

```


* **Upserting Data (Insert or Update on Conflict):**
If a row with a unique key already exists, PostgreSQL can update it instead of failing:
```sql
INSERT INTO chicago_dmv.Vehicle (VEHICLE_ID, VEHICLE_MAKE, VEHICLE_YEAR)
VALUES (55, 'Honda', 2021)
ON CONFLICT (VEHICLE_ID) DO UPDATE SET 
    VEHICLE_MAKE = excluded.VEHICLE_MAKE,
    VEHICLE_YEAR = excluded.VEHICLE_YEAR;

```



---

## **5. Querying Data (DQL)**

The `SELECT` statement allows you to pull and filter data.

### **Filtering (`WHERE`) and Sorting (`ORDER BY`)**

```sql
SELECT VEHICLE_MAKE, VEHICLE_YEAR
FROM chicago_dmv.Vehicle
WHERE VEHICLE_YEAR >= 2015
ORDER BY VEHICLE_YEAR DESC
LIMIT 10;

```

### **Aggregations & Grouping (`GROUP BY` / `HAVING`)**

To find the total number of crashes per vehicle make:

```sql
SELECT VEHICLE_MAKE, COUNT(*) AS total_vehicles
FROM chicago_dmv.Vehicle
GROUP BY VEHICLE_MAKE
HAVING COUNT(*) > 5
ORDER BY total_vehicles DESC;

```

---

## **6. Joining Relational Tables**

Because relational databases split data across multiple tables (normalized design), you use **Joins** to connect them using shared identifiers (like `CRASH_ID` or `VEHICLE_ID`).

* **Inner Join (Returns only matching rows from both tables):**
```sql
SELECT p.PERSON_ID, p.PERSON_AGE, v.VEHICLE_MAKE, v.VEHICLE_YEAR
FROM chicago_dmv.Person p
INNER JOIN chicago_dmv.Vehicle v 
    ON p.VEHICLE_ID = v.VEHICLE_ID;

```


* **Left Join (Returns all records from the left table, and matching ones from the right):**
```sql
SELECT c.CRASH_ID, t.CRASH_HOUR, c.TOTAL_INJURIES
FROM chicago_dmv.Crash c
LEFT JOIN chicago_dmv.Crash_Time t 
    ON c.CRASH_ID = t.CRASH_ID;

```

---

## **Quick Reference Summary**

| Command Type | Action | SQL Example |
| --- | --- | --- |
| **DDL** | Create Table | `CREATE TABLE name (col type);` |
| **DDL** | Drop Table | `DROP TABLE name;` |
| **DML** | Insert Data | `INSERT INTO name (col) VALUES (val);` |
| **DML** | Update Data | `UPDATE name SET col = val WHERE condition;` |
| **DQL** | Select Data | `SELECT col FROM name WHERE condition;` |
| **Meta** | Connect DB | `\c database_name` |

## **Part 1: PostgreSQL Users & Roles Management**

### **1. Create a New User**

You can create a user with a secure password using `CREATE USER` or `CREATE ROLE`.

```sql
CREATE USER data_analyst WITH PASSWORD 'SuperSecurePassword123';
```

*(If you want the user to be able to create other databases or roles, you can add flags like `CREATEDB` or `CREATEROLE`.)*

### **2. List All Users**

Inside the `psql` shell, use the meta-command to view all users and their roles:

```sql
\du

```

### **3. Modify a User's Password or Attributes**

If you need to change a password or update permissions later:

```sql
ALTER USER data_analyst WITH PASSWORD 'NewPassword456';
```

### **4. Delete a User**

To remove a user from the database system:

```sql
DROP USER data_analyst;
```


## **Part 2: Schema Management**

A **schema** is a named collection of tables, views, and functions. By default, PostgreSQL puts everything in a schema named `public`. Creating custom schemas helps keep your project organized (e.g., `chicago_dmv`, `finance`, `hr`).

### **1. Create a Schema and Assign an Owner**

You can create a schema and immediately assign ownership to a specific user:

```sql
CREATE SCHEMA chicago_dmv AUTHORIZATION data_analyst;

```

Or simply:

```sql
CREATE SCHEMA IF NOT EXISTS chicago_dmv;

```

### **2. List All Schemas**

To view all schemas available in your current database:

```sql
\dn

```

### **3. Drop a Schema**

If you want to delete an entire schema along with all the tables inside it (use with caution!):

```sql
DROP SCHEMA chicago_dmv CASCADE;

```

*(The `CASCADE` keyword ensures that all tables inside the schema are dropped too; otherwise, PostgreSQL will block the deletion if objects still exist inside it.)*

## **Part 3: Managing Privileges (GRANT & REVOKE)**

Just creating a user and a schema isn't enough—by default, a new user often has no permissions to view or write data in custom schemas. You must explicitly grant access using **DCL (Data Control Language)** commands.

### **1. Grant Schema Access**

A user needs `USAGE` permission on a schema just to see or access the tables inside it:

```sql
GRANT USAGE ON SCHEMA chicago_dmv TO data_analyst;

```

### **2. Grant Table Permissions**

You can grant specific permissions (`SELECT`, `INSERT`, `UPDATE`, `DELETE`) on existing or future tables:

```sql
-- Grant read-only access to a specific table
GRANT SELECT ON chicago_dmv.Vehicle TO data_analyst;

-- Grant full read/write access to all tables in the schema
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA chicago_dmv TO data_analyst;
```

### **3. Revoke Permissions**

If you need to take away a permission you previously granted:

```sql
REVOKE INSERT ON chicago_dmv.Vehicle FROM data_analyst;
```

## **Part 4: Essential Administrative `psql` Commands**

Here is a quick reference table of terminal commands used during database and user administration:

| Command | Description |
| --- | --- |
| `\du` | List all users/roles and their attributes |
| `\l` | List all databases on the server |
| `\dn` | List all schemas in the current database |
| `\dt` | List all tables in the current schema/database |
| `\d table_name` | Describe a table's structure (columns, types, keys) |
| `\c db_name` | Switch to a different database |
| `\q` | Exit the `psql` terminal |